# Conversatioanl RAG Apllication with LangChainand OpenAILLm

In [ ]:
# Install the necessary packages
! pip install langchain -qU --upgrade --force-reinstall
!pip install langchain-google-genai -qU --upgrade --force-reinstall
! pip install chromadb -qU --upgrade --force-reinstall
! pip install pypdf -q --upgrade --force-reinstall
! pip install langchain-community -qU --upgrade --force-reinstall
! pip install langchain-chroma -qU --upgrade --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata

# Initialize GeminiAI LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from google.colab import userdata


if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Initialize model for token counting
llm_for_token_count = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

# Initialize Embedding Model

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding_model = GoogleGenerativeAIEmbeddings(model= "models/gemini-embedding-001")

# Load PDF Document

In [ ]:
! pip install pypdf -q

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF document
loader = PyPDFLoader ("/content/codeprolk.pdf")

docs = loader.load()

In [ ]:
len(docs)

4

In [ ]:
docs[0]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-18T10:00:42+05:30', 'author': 'Dinesh Piyasamara', 'moddate': '2024-06-18T10:00:42+05:30', 'source': '/content/codeprolk.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content="Introduction to CodePRO LK \nCodePRO LK is a dynamic educational platform that offers a diverse range of technology-\nrelated courses in Sinhala, aimed at empowering Sri Lankans with valuable skills in \nprogramming, data science, and machine learning. Founded by Dinesh Piyasamara during the \nCOVID-19 pandemic, CodePRO LK addresses the growing need for accessible, high-quality \ntech education tailored to the local community. \n \nFounding and Vision \nOrigin and Motivation \nThe inception of CodePRO LK was driven by the challenges posed by the COVID-19 pandemic, \nwhich highlighted the necessity for remote learning and digital skills. Recognizing this, Dinesh

# Splits Documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter (chunk_size = 400, chunk_overlap =50)

# Split the documents into chunks
splits = text_splitter.split_documents(docs)

In [ ]:
len(splits)

20

In [ ]:
splits[0]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-18T10:00:42+05:30', 'author': 'Dinesh Piyasamara', 'moddate': '2024-06-18T10:00:42+05:30', 'source': '/content/codeprolk.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Introduction to CodePRO LK \nCodePRO LK is a dynamic educational platform that offers a diverse range of technology-\nrelated courses in Sinhala, aimed at empowering Sri Lankans with valuable skills in \nprogramming, data science, and machine learning. Founded by Dinesh Piyasamara during the \nCOVID-19 pandemic, CodePRO LK addresses the growing need for accessible, high-quality')

# Create Vector Store and Retriever

In [ ]:
from langchain_chroma import Chroma

# Create a vector store from the document chunks
vectorstore = Chroma.from_documents  (documents = splits, embedding = embedding_model)

In [ ]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

# Define Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define the system propmt
system_prompt = (
    "You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer,just say that you dont't know"
    "/n/n"
    "{context}"
)

# Create the prompt template
prompt= ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)


In [ ]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer,just say that you dont't know/n/n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

# Create Retrieval_AUgmented Generation (RAG) Chain

In [ ]:
from langchain.chains import create_retrieval_chain, create_stuff_documents_chain

ModuleNotFoundError: No module named 'langchain.chains'